In [1]:
import pandas as pd
import arviz as az
from Functions import *

In [2]:
data=pd.read_excel(r"../Data_and_Preprocessing/before_season_correction.xlsx")
device='cuda'

In [4]:
data

,Unnamed: 0.1,Unnamed: 0,corrected_week,gastove,passive,road,household,no2,nox,formaldehyd,acetald,acetone,Total_VOCs
0,0,0,13.116186,0.779790,705.692679,25809.494379,89,11.914562,5.440693,9.924185,11.781606,15.080416,36.786207
1,1,1,-19.871429,16.340792,365.061947,19621.955554,204,10.819796,9.372262,16.884841,1.225847,9.594748,27.705436
2,2,2,18.610158,3.771551,0.289160,15945.489134,240,4.545942,4.931943,27.141280,NaN,NaN,27.141280
3,3,3,-12.036755,0.715447,0.000000,18643.976634,138,7.474815,12.906349,17.822297,1.508872,5.389186,24.720355
4,4,4,10.702850,0.381364,0.000000,20963.228891,129,7.961357,7.746275,28.859281,7.124666,10.886205,46.870152
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1267,1267,1267,41.843000,48.431324,732.484276,19510.836756,223,4.190477,16.187692,42.176797,1.632585,2.564476,46.373858
1268,1268,1268,4.947519,0.069941,0.197983,18286.633266,140,7.155577,26.520427,11.442885,11.422987,10.546744,33.412616
1269,1269,1269,23.318083,362.279702,174.555450,9830.672218,238,8.364410,9.509836,17.271125,13.619011,15.258063,46.148199
1270,1270,1270,35.971961,310.490627,0.453096,18792.958431,256,5.626440,15.381738,12.677648,NaN,8.919656,21.597304


In [3]:
train=data.loc[data.label==0,:]

AttributeError: 'DataFrame' object has no attribute 'label'

In [4]:
train=make_periodic(train,"corrected_week",device ) 

/home/michaelf/Seasonality_pyro_rewrite/Seasonality_functions/Functions.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.loc[:,"theta"]=data.loc[:,variable]/data.loc[:,variable].max()*2*np.pi
/home/michaelf/Seasonality_pyro_rewrite/Seasonality_functions/Functions.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.loc[:,"cos_theta"]=np.cos(data.loc[:,"theta"])
/home/michaelf/Seasonality_pyro_rewrite/Seasonality_functions/Functions.py:18: SettingWithCopyWarning: 
A value is trying to be set on

In [5]:
X_no2,y_no2=make_Xy(train,["cos_theta","sin_theta"],"no2",device) 
nuts_kernel_no2,gpr_no2=model(X_no2,y_no2,device) 
pyro_data,gpr_no2=train_model(X_no2,nuts_kernel_no2,gpr_no2,device) 
torch.save(gpr_no2, "models/no2_seasonality");

Sample: 100%|██████████████████████████████████████| 14000/14000 [14:14, 16.38it/s, step size=7.00e-01, acc. prob=0.931]
/data/michaelf/miniconda3/lib/python3.10/site-packages/arviz/data/io_pyro.py:158: UserWarning: Could not get vectorized trace, log_likelihood group will be omitted. Check your model vectorization or set log_likelihood=False
  warnings.warn(


In [6]:
X_nox,y_nox=make_Xy(train,["cos_theta","sin_theta"],"nox",device) 
nuts_kernel,gpr_nox=model(X_nox,y_nox,device) 
pyro_data,gpr_nox=train_model(X_nox,nuts_kernel,gpr_nox,device) 
torch.save(gpr_nox, "models/nox_seasonality");

Sample: 100%|██████████████████████████████████████| 14000/14000 [07:36, 30.67it/s, step size=8.45e-01, acc. prob=0.899]
/data/michaelf/miniconda3/lib/python3.10/site-packages/arviz/data/io_pyro.py:158: UserWarning: Could not get vectorized trace, log_likelihood group will be omitted. Check your model vectorization or set log_likelihood=False
  warnings.warn(
